<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-11-self-hosting/lesson-11.1-gemma-cloud-run/notebooks/GCP_Capstone_11.1_GemmaCloudRun.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.1 Gemma on Cloud Run L4 — The Bill Before the Build
**Netsetos GenAI Engineering — GCP Capstone** · Module 11 · rebuilt on the live lane, 10 September 2026

A GPU on Cloud Run is a bill with a model behind it. This lesson prices the instance before anything is built (the L4 is less than half of it), reads the project's GPU quota through the API that holds it, builds 11.2's server from the kit's own Cloud Build config behind a switch, explains `make deploy-vllm` flag by flag, waits for the real service the way a client must, asks it one question through an OpenAI-compatible door with a Google ID token, and feeds the cost calculator with the lane's own tokens per day. Every number is the lane's or a rate table's; nothing is typed in.


## Setup
The kit, the roster member, the lane's URLs - the gateway's and the SLM's derived from the project number like every other service's.


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
BUILD    = False   # True submits the Cloud Build of the 13 GB image: 20-30 min, and Gemma access in the hf-token secret
VLLM_URL = f"https://documind-vllm-{NUMBER}.{REGION}.run.app"   # make deploy-vllm (decision D3: optional on the lane)

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers):
    the body's `model` names what answered (a fallback included), the headers carry the cost the gateway priced."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def slm(path: str, body: dict | None = None, method: str = "POST", timeout: int = 240) -> tuple[int, dict | str, float]:
    """One call to the SLM's own doors (Ollama's /api/*, or its OpenAI-compatible /v1/*), timed - the first call after
    idle is the cold start."""
    t0 = time.time()
    r = requests.request(method, f"{SLM_URL}{path}", json=body, timeout=timeout,
                         headers={"Authorization": f"Bearer {documind_tools._id_token(SLM_URL)}"})
    try:
        return r.status_code, r.json(), time.time() - t0
    except ValueError:
        return r.status_code, r.text[:400], time.time() - t0

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: run_eval, judge
sys.path.insert(0, f"{KIT}/deploy/services/slm")        # compare_backends, make_modelfile
print("helpers: api(), gateway(), slm(), service(), usage_rows(); the kit's evals/ and services/slm/ on sys.path")


## Cell 2: The bill first
Three rates, one instance, and the flags read from the kit's Makefile so the table prices what is actually deployed.


In [ ]:
# THE BILL FIRST. The L4 is the line everyone quotes and less than half of what the instance costs: the deploy asks
# for 8 vCPU and 32 GiB beside it, and Cloud Run bills all three for every second the instance is up. The rates are
# us-central1's (verified 4 September 2026); the flags are read from the kit's own Makefile, so the table prices the
# instance make deploy-vllm actually creates - not one from a slide.
USD_INR = 85
RATES = [("L4 GPU (no zonal redundancy)", 0.672), ("CPU (8 vCPU)", 0.518), ("Memory (32 GiB)", 0.230)]
mk = open(f"{KIT}/deploy/Makefile", encoding="utf-8").read()
DEPLOY_VLLM = mk.split("deploy-vllm:", 1)[1].split(chr(10) * 2, 1)[0]
assert "--gpu 1 --gpu-type nvidia-l4 --no-gpu-zonal-redundancy --cpu 8 --memory 32Gi" in DEPLOY_VLLM, "the deploy's shape changed: re-price it"
print(f"  {'component':34} {'per hour':>9} {'per month':>11} {'INR/month':>12}")
INSTANCE_HR = 0.0
for label, rate in RATES:
    INSTANCE_HR += rate
    print(f"  {label:34} ${rate:>8.3f} ${rate * 24 * 30:>10,.0f} Rs {rate * 24 * 30 * USD_INR:>9,.0f}")
print(f"  {'TOTAL INSTANCE (make deploy-vllm)':34} ${INSTANCE_HR:>8.2f} ${INSTANCE_HR * 24 * 30:>10,.0f} Rs {INSTANCE_HR * 24 * 30 * USD_INR:>9,.0f}")
print()
print("  ... for every hour it is up. --min-instances 0 makes the idle rate zero; a warm instance is the whole table, every month,")
print("  and 11.4 shows the break-even that follows: about 379,000 answers a month before this is cheaper than Gemini.")


## Cell 3: The quota, before the build


In [ ]:
# THE QUOTA, BEFORE THE BUILD. Cloud Run GPUs are a Service Usage quota on run.googleapis.com, per region - not a
# Compute Engine one, so `gcloud compute` cannot see it. Read what this project may allocate before spending half an
# hour on an image it cannot run: us-central1 self-serves a small default on new projects, asia-south1 is invitation-
# only (11.4's table). A refused deploy costs nothing; a refused deploy after a 25-minute build costs 25 minutes.
sess = AuthorizedSession(creds)
r = sess.get(f"https://serviceusage.googleapis.com/v1beta1/projects/{NUMBER}/services/run.googleapis.com/consumerQuotaMetrics",
             params={"pageSize": 500, "view": "BASIC"}, timeout=60)
metrics = r.json().get("metrics", []) if r.status_code == 200 else []
gpu = [m for m in metrics if "gpu" in (m.get("metric", "") + m.get("displayName", "")).lower()]
print(f"run.googleapis.com quota metrics: {len(metrics)} (HTTP {r.status_code}); GPU-shaped: {len(gpu)}")
for m in gpu:
    for lim in m.get("consumerQuotaLimits", []):
        for b in lim.get("quotaBuckets", []):
            dims = b.get("dimensions") or {}
            if dims.get("region", REGION) in (REGION, "asia-south1"):
                print(f"  {m.get('displayName', m['metric'])[:58]:58} {str(dims or 'default'):26} limit {b.get('effectiveLimit', '?')}")
if not gpu:
    print("  no GPU quota metric is visible to this account: the first `make deploy-slm` is the quota test, and a refusal is free")


## Cell 4: The image is the kit's
`services/gemma-vllm/cloudbuild.yaml`, the `hf-token` secret, and the build behind `BUILD`.


In [ ]:
# THE IMAGE IS THE KIT'S. services/gemma-vllm/cloudbuild.yaml builds 11.2's server on the vLLM base with the Gemma
# weights baked in - so HF_HUB_OFFLINE=1 is correct and a cold start pulls an image, not the internet. The token comes
# from Secret Manager under the name secrets.tf creates, hf-token (the secret 10.5 pushes with), never from the repo.
# The build sits behind BUILD: 20 to 30 minutes on E2_HIGHCPU_32, a 13 GB push, and a Hugging Face account that has
# accepted the Gemma licence - without that it fails at the download, after the base image has already been pulled.
cb = open(f"{KIT}/deploy/services/gemma-vllm/cloudbuild.yaml", encoding="utf-8").read()
print(chr(10).join(l for l in cb.splitlines() if not l.startswith("#")))
assert "secrets/hf-token/versions/latest" in cb and "${_IMAGE}" in cb, "the build must read hf-token and push to _IMAGE"
versions = subprocess.run(["gcloud", "secrets", "versions", "list", "hf-token", "--project", PROJECT_ID, "--filter=state=ENABLED",
                           "--format=value(name)"], capture_output=True, text=True).stdout.split()
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/documind/gemma-vllm:latest"
print(f"hf-token: {len(versions)} enabled version(s) | image: {IMAGE}")
if BUILD and versions:
    r = subprocess.run(["gcloud", "builds", "submit", "services/gemma-vllm", "--project", PROJECT_ID, "--config",
                        "services/gemma-vllm/cloudbuild.yaml", f"--substitutions=_IMAGE={IMAGE}"], cwd=f"{KIT}/deploy", text=True)
    print("build exit:", r.returncode)
elif BUILD:
    print("BUILD is True but hf-token has no enabled version: add one (a token whose account accepted the Gemma licence) and re-run")
else:
    print(f"BUILD is False. In Cloud Shell: make build-vllm PROJECT={PROJECT_ID}   (then make deploy-vllm; decision D3)")


## Cell 5: The deploy, flag by flag - and the poller against the real URL


In [ ]:
# THE DEPLOY, FLAG BY FLAG. make deploy-vllm is the command; every flag is a bill or a failure mode, and the same six
# cost flags carry 11.4's SLM. Then, if the service exists, the health poller waits for it the way a client must: a
# GPU service reports healthy only once the weights are on the card, and the first request after idle is a cold start.
print(chr(10).join(l for l in DEPLOY_VLLM.splitlines() if l.strip().startswith(("gcloud run deploy", "--"))))
FLAGS = [
    ("--gpu 1 --gpu-type nvidia-l4", "one L4, lowercase l: nvidia-L4 is rejected"),
    ("--no-gpu-zonal-redundancy", "$0.672/hr instead of $1.047: no failover across zones, which one instance never had"),
    ("--cpu 8 --memory 32Gi", "the recommended shape for an L4, and the $0.748/hr the GPU line hides"),
    ("--max-instances 1 --min-instances 0", "each instance is a GPU; zero idle is what makes the bill a duty cycle"),
    ("--no-cpu-throttling --cpu-boost", "vLLM needs CPU between requests; the boost cuts minutes off the start"),
    ("--no-allow-unauthenticated", "IAM at the door: the gateway's and the UI's accounts are invited, nobody else"),
    ("--startup-probe /health, 120 s + 5 x 30 s", "the weights load before traffic arrives; without it the first request meets an empty engine"),
]
print()
for f, why in FLAGS:
    print(f"  {f:44} {why}")

def wait_for_ready(url: str, max_attempts: int = 20, base_delay: float = 2.0) -> bool:
    """Poll /health with backoff - a client's cold-start discipline. 200 only once the engine is loaded."""
    for attempt in range(max_attempts):
        try:
            r = requests.get(f"{url}/health", headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=10)
            if r.status_code == 200 and r.json().get("status") == "ok":
                print(f"  ready after {attempt + 1} attempt(s): {r.json()}")
                return True
            print(f"  attempt {attempt + 1}: HTTP {r.status_code} {r.text[:60]!r}")
        except requests.RequestException as e:
            print(f"  attempt {attempt + 1}: {type(e).__name__}")
        time.sleep(min(base_delay * (1.5 ** attempt), 30.0))
    return False

vllm = service("documind-vllm")
if vllm:
    print(f"\ndocumind-vllm: image {vllm['image'].rsplit('/', 1)[-1]} | min-instances {vllm['min_instances']}")
    READY = wait_for_ready(VLLM_URL)
    assert READY, "the engine never reported ready: read the service's logs (the weights, the GPU, the probe)"
else:
    READY = False
    print(f"\ndocumind-vllm is not deployed (decision D3). make build-vllm && make deploy-vllm PROJECT={PROJECT_ID} -")
    print("or the gateway's documind-inference route serves the SLM's OpenAI-compatible door instead (next cell)")


## Cell 6: The client
One asserted completion through whichever OpenAI-compatible door the lane has, with an ID token minted per call.


In [ ]:
# THE CLIENT. An OpenAI-compatible door behind Cloud Run IAM: the SDK needs a base_url, a placeholder api_key and one
# header - a Google ID token for the service's URL, minted as the roster member, per call (7.3's hour-long fuse never
# arms; the kit mints it). The lane has that door twice: the engine's own /v1 when 11.1's image is deployed, and the
# gateway's documind-inference route, which on the lean lane is the SLM's /v1 - Ollama speaks the same API. The route
# stays; the engine behind it is a deploy.
if READY:
    DOOR, MODEL, AUD = f"{VLLM_URL}/v1", "google/gemma-3-4b-it", VLLM_URL
else:
    DOOR, MODEL, AUD = f"{GATEWAY_URL}/v1", "documind-inference", GATEWAY_URL
print(f"door: {DOOR}  model: {MODEL}")
print('the SDK shape: OpenAI(base_url=DOOR, api_key="unused", default_headers={"Authorization": f"Bearer {documind_tools._id_token(AUD)}"})')
t0 = time.time()
r = requests.post(f"{DOOR}/chat/completions", timeout=240, headers={"Authorization": f"Bearer {documind_tools._id_token(AUD)}"},
                  json={"model": MODEL, "max_tokens": 8, "temperature": 0.0,
                        "messages": [{"role": "system", "content": "Classify the document as INVOICE, CONTRACT, REPORT or LETTER. Reply with ONE word."},
                                     {"role": "user", "content": "This Master Services Agreement is made between ACME Corp and the Supplier, effective 1 April 2026 ..."}]})
secs = time.time() - t0
assert r.status_code == 200, f"HTTP {r.status_code}: {r.text[:200]}"
body = r.json()
text = body["choices"][0]["message"]["content"].strip()
print(f"answer: {text!r} from {body.get('model')!r} in {secs:.1f}s (the first call after idle is the cold start; the second is the model)")
assert "CONTRACT" in text.upper(), f"a one-word classification was asked for; got {text!r}"


## Cell 7: The calculator, with the lane's inputs


In [ ]:
# THE CALCULATOR, WITH THE LANE'S INPUTS. 11.1's arithmetic - an instance-hour against Gemini's per-token rates - fed
# with what the lane actually does: tokens per day from the usage rows, the in/out split as measured. The table scales
# that up, and the honest reading is the top row: at the lane's own volume the instance is the dearer option by a wide
# margin, and only 10.5's reasons - residency and format - justify it. 11.4 turns this into a break-even in answers.
FLASH_IN, FLASH_OUT, PRO_IN, PRO_OUT = 1.50, 7.50, 2.00, 12.00      # USD per 1M tokens, standard rates (CLAUDE.md)
rows = usage_rows(minutes=60 * 24, limit=1000)
tok_in, tok_out = sum(r.get("tokens_in", 0) for r in rows), sum(r.get("tokens_out", 0) for r in rows)
LANE_PER_DAY = tok_in + tok_out
share_out = tok_out / LANE_PER_DAY if LANE_PER_DAY else 0.2
print(f"the lane, last 24 h: {len(rows)} answers, {LANE_PER_DAY:,} tokens ({share_out:.0%} output)")

def cloud_run_usd(hours_per_day: float, instances: int = 1) -> float:
    return INSTANCE_HR * hours_per_day * 30 * instances

def gemini_usd(tokens_per_day: int, rate_in: float, rate_out: float) -> float:
    t = tokens_per_day * 30
    return (t * (1 - share_out) * rate_in + t * share_out * rate_out) / 1_000_000

print(f"\n{'tokens/day':>14} {'Cloud Run L4, 8 h/day':>22} {'Gemini flash':>13} {'Gemini pro':>11}   cheaper")
for volume in sorted({max(LANE_PER_DAY, 1), 2_000_000, 10_000_000, 50_000_000, 100_000_000}):
    instances = max(1, volume // 25_000_000)
    cr, gf = cloud_run_usd(8, instances), gemini_usd(volume, FLASH_IN, FLASH_OUT)
    tag = "  <- the lane today" if volume == max(LANE_PER_DAY, 1) else ""
    print(f"{volume:>14,} ${cr:>21,.0f} ${gf:>12,.0f} ${gemini_usd(volume, PRO_IN, PRO_OUT):>10,.0f}   {'self-hosted' if cr < gf else 'gemini'}{tag}")
print("\n8 warm hours a day; 24 is three times the Cloud Run column. 11.5 measures the lane's duty cycle instead of assuming one.")

PATTERNS = [
    ("scale-to-zero batch", "the night's uploads classified at 02:00", "min 0 / max 5 / concurrency 64", "$0 idle; a 19-35 s cold start nobody is waiting for"),
    ("warm interactive", "user-facing Q&A that must not cold start", "min 1 / max 3 / concurrency 32", f"Rs {INSTANCE_HR * 24 * 30 * USD_INR:,.0f} a month, every month; Gemini on overflow"),
    ("one L4, several models", "classify -> extract -> summarise on one card", "2 + 8 + 9 GB of a 24 GB L4", "one instance instead of three endpoints; one failure domain"),
]
print(f"\n{'pattern':24} {'use':42} {'config':32} cost and trade")
for name, use, cfg, cost in PATTERNS:
    print(f"{name:24} {use:42} {cfg:32} {cost}")


## Where this goes
- **11.2** reads the server this image runs - seven files in the kit, and the two doors a request needs.
- **11.3** puts a route in front of every door; **11.4** puts 10.5's model behind one and prices it in answers; **11.5** measures the duty cycle this calculator assumed.

## ✅ Lesson 11.1 complete
- ✅ The instance priced before the build: three rates, one bill, and the flags that make it so
- ✅ The GPU quota read through Service Usage, per region, before anything was built
- ✅ The kit's Cloud Build with the secret Terraform makes, behind a switch
- ✅ `make deploy-vllm` flag by flag, the poller against the real service when it exists
- ✅ One asserted completion through an OpenAI-compatible door with a per-call ID token
- ✅ The calculator fed with the lane's own tokens per day; the three patterns as a table
